In [14]:
import random

# Base directories
SEM_BASE = "/home/acevedo/syn-sem/datasets/txt/sem/second/matching"
SYN_BASE = "/home/acevedo/syn-sem/datasets/txt/syn/second/matching"

# Semantic files
SEM_ENGLISH_FILE = f"{SEM_BASE}/english/sentences0.txt"
SEM_PARAPHRASE_FILE = f"{SEM_BASE}/english/sentences1.txt"
SEM_INDICES_FILE = f"{SEM_BASE}/english/sem_ids_with_syn.txt"

# Other semantic language files
SEM_LANGUAGE_FILES = {
    "chinese": f"{SEM_BASE}/chinese/sentences1.txt",
    "spanish": f"{SEM_BASE}/spanish/sentences1.txt",
    "italian": f"{SEM_BASE}/italian/sentences1.txt",
    "turkish": f"{SEM_BASE}/turkish/sentences1.txt",
    "german":  f"{SEM_BASE}/german/sentences1.txt",
    "arabic":  f"{SEM_BASE}/arabic/sentences1.txt"
}

# Syntax files
SYN_ENGLISH_FILE = f"{SYN_BASE}/english/sentences0.txt"
GROUP_IDS_FILE = f"{SYN_BASE}/english/group_ids.txt"
SYNTAX_MATCHED_FILE = f"{SYN_BASE}/english/syntax_matched_sentences.txt"


def pick_random_sentence(sem_eng_file, sem_paraphrase_file, sem_ids_file, sem_lang_files, syn_eng_file, group_file):
    """Pick a random semantic sentence, get paraphrase, translations, syntax ID and group ID."""
    
    # Load semantic English sentences
    with open(sem_eng_file, "r", encoding="utf-8") as f:
        sem_eng_sentences = [line.strip() for line in f]

    # Load paraphrases
    with open(sem_paraphrase_file, "r", encoding="utf-8") as f:
        paraphrases = [line.strip() for line in f]

    # Load semantic indices
    with open(sem_ids_file, "r", encoding="utf-8") as f:
        indices = [int(line.strip()) for line in f if line.strip().isdigit()]

    if not indices:
        raise ValueError("No valid indices found in semantic index file.")

    # Pick random index
    chosen_index = random.choice(indices)
    sentence = sem_eng_sentences[chosen_index]
    paraphrase_sentence = paraphrases[chosen_index] if chosen_index < len(paraphrases) else "[Paraphrase not found]"

    # Collect translations
    translations = {"english": sentence, "english_paraphrase": paraphrase_sentence}
    for lang, path in sem_lang_files.items():
        with open(path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f]
            translations[lang] = lines[chosen_index] if chosen_index < len(lines) else "[Index out of range]"

    # Load syntax sentences
    with open(syn_eng_file, "r", encoding="utf-8") as f:
        syn_sentences = [line.strip() for line in f]

    # Find syntax ID
    try:
        syn_id = syn_sentences.index(sentence)
    except ValueError:
        syn_id = None

    # Load group IDs
    with open(group_file, "r", encoding="utf-8") as f:
        group_ids = [line.strip() for line in f]

    group_id = int(group_ids[syn_id]) if syn_id is not None and syn_id < len(group_ids) else "[Group ID not found]"

    return chosen_index, translations, syn_id, group_id


def print_syntax_sentences(group_id, syn_eng_file, group_file, max_lines=5):
    """Print up to max_lines syntax sentences sharing the same group_id."""
    
    with open(group_file, "r", encoding="utf-8") as f:
        group_ids = [line.strip() for line in f]

    matching_indices = [i for i, gid in enumerate(group_ids) if gid == str(group_id)]
    if not matching_indices:
        print(f"No sentences found with group_id {group_id}")
        return

    matching_indices = matching_indices[:max_lines]

    with open(syn_eng_file, "r", encoding="utf-8") as f:
        syn_sentences = [line.strip() for line in f]

    print(f"\n--- Sentences with the same group_id ({group_id}) ---")
    for idx in matching_indices:
        if idx < len(syn_sentences):
            print(f"- {syn_sentences[idx]}")
        else:
            print(f"- [Index {idx} out of range]")


def print_syntax_pos(syntax_sentence, syntax_file):
    """Search for the syntax_sentence in the TAB-separated file and print its POS structure (3rd column)."""
    with open(syntax_file, "r", encoding="utf-8") as f:
        for line in f:
            columns = line.strip().split("\t")
            if len(columns) >= 3 and columns[0] == syntax_sentence:
                pos_structure = columns[2]
                print(f"\nPOS structure of the syntax example:\n{pos_structure}")
                return
    print("\nPOS structure not found for the syntax example.")


if __name__ == "__main__":
    idx, sentences, syn_id, group_id = pick_random_sentence(
        SEM_ENGLISH_FILE, SEM_PARAPHRASE_FILE, SEM_INDICES_FILE, SEM_LANGUAGE_FILES,
        SYN_ENGLISH_FILE, GROUP_IDS_FILE
    )

    # Print English sentence and paraphrase
    print(f"Chosen semantic index: {idx}\n")
    print(f"English: {sentences['english']}")
    print(f"English paraphrase: {sentences['english_paraphrase']}\n")

    # Print translations in other languages
    for lang in SEM_LANGUAGE_FILES.keys():
        print(f"{lang.capitalize()}: {sentences[lang]}")

    if group_id != "[Group ID not found]":
        print_syntax_sentences(group_id, SYN_ENGLISH_FILE, GROUP_IDS_FILE, max_lines=5)

    # Print POS structure of the syntax example
    if syn_id is not None:
        print_syntax_pos(sentences['english'], SYNTAX_MATCHED_FILE)


Chosen semantic index: 1096

English: Many viewers prefer a happy ending
English paraphrase: A happy ending is preferred by many viewers

Chinese: 许多观众更喜欢圆满结局
Spanish: Muchos espectadores prefieren un final feliz
Italian: Molti spettatori preferiscono un lieto fine
Turkish: Birçok izleyici mutlu sonu tercih eder
German: Viele Zuschauer bevorzugen ein Happy End
Arabic: يفضّل كثير من المشاهدين نهاية سعيدة

--- Sentences with the same group_id (45) ---
- His ideas challenge every conventional thought
- Those trees provide a welcome shade
- Many chefs use a secret ingredient
- Few politicians tell the whole truth
- Many viewers prefer a happy ending

POS structure of the syntax example:
DT NNS VBP DT JJ NN


In [16]:
import random
import re

# --- File paths ---
SEM_BASE = "/home/acevedo/syn-sem/datasets/txt/sem/second/matching"
SYN_BASE = "/home/acevedo/syn-sem/datasets/txt/syn/second/matching"

SEM_ENGLISH_FILE = f"{SEM_BASE}/english/sentences0.txt"
SEM_PARAPHRASE_FILE = f"{SEM_BASE}/english/sentences1.txt"
SEM_INDICES_FILE = f"{SEM_BASE}/english/sem_ids_with_syn.txt"

SEM_LANGUAGE_FILES = {
    "chinese": f"{SEM_BASE}/chinese/sentences1.txt",
    "spanish": f"{SEM_BASE}/spanish/sentences1.txt",
    "italian": f"{SEM_BASE}/italian/sentences1.txt",
    "turkish": f"{SEM_BASE}/turkish/sentences1.txt",
    "german":  f"{SEM_BASE}/german/sentences1.txt",
    "arabic":  f"{SEM_BASE}/arabic/sentences1.txt"
}

SYN_ENGLISH_FILE = f"{SYN_BASE}/english/sentences0.txt"
GROUP_IDS_FILE   = f"{SYN_BASE}/english/group_ids.txt"
SYNTAX_MATCHED_FILE = f"{SYN_BASE}/english/syntax_matched_sentences.txt"


# --- LaTeX helpers ---
def escape_latex(text):
    """Escape LaTeX special characters."""
    replacements = {
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\^{}",
        "\\": r"\textbackslash{}"
    }
    for key, val in replacements.items():
        text = text.replace(key, val)
    return text


def wrap_cjk(text):
    """Wrap Chinese characters in CJK environment."""
    if re.search(r'[\u4E00-\u9FFF]', text):
        return r"\begin{CJK}{UTF8}{gbsn}" + text + r"\end{CJK}"
    return text


# Simple Arabic transliteration for arabtex
arabic_translit_map = {
    "ا": "A", "ب": "b", "ت": "t", "ث": "v", "ج": "j", "ح": "H", "خ": "x",
    "د": "d", "ذ": "*", "ر": "r", "ز": "z", "س": "s", "ش": "$", "ص": "S",
    "ض": "D", "ط": "T", "ظ": "Z", "ع": "E", "غ": "g", "ف": "f", "ق": "q",
    "ك": "k", "ل": "l", "م": "m", "ن": "n", "ه": "h", "و": "w", "ي": "y",
    "ء": "'", "أ": "A", "إ": "A", "ؤ": "w", "ئ": "y", "ى": "A", "ة": "h"
}

def arabic_to_arabtex(text):
    """Transliterate Arabic characters to arabtex ASCII."""
    return "".join(arabic_translit_map.get(c, c) for c in text)

def wrap_arabic(text):
    """Wrap Arabic in \RL{} for pdfLaTeX with arabtex.

    This assumes `arabic_to_arabtex` produces proper ArabTeX transliteration.
    Do NOT escape $ or other symbols — they are part of ArabTeX.
    """
    translit = arabic_to_arabtex(text)
    return r"\RL{" + translit + "}"



def wrap_multilingual(text, lang):
    """Wrap Chinese or Arabic as needed."""
    if lang.lower() == "chinese":
        return wrap_cjk(text)
    elif lang.lower() == "arabic":
        return wrap_arabic(text)
    else:
        return escape_latex(text)


# --- Core functions ---
def pick_random_sentence(sem_eng_file, sem_paraphrase_file, sem_ids_file, sem_lang_files, syn_eng_file, group_file):
    """Pick a random semantic sentence, get paraphrase, translations, syntax ID and group ID."""
    with open(sem_eng_file, "r", encoding="utf-8") as f:
        sem_eng_sentences = [line.strip() for line in f]
    with open(sem_paraphrase_file, "r", encoding="utf-8") as f:
        paraphrases = [line.strip() for line in f]
    with open(sem_ids_file, "r", encoding="utf-8") as f:
        indices = [int(line.strip()) for line in f if line.strip().isdigit()]
    if not indices:
        raise ValueError("No valid indices found in semantic index file.")

    chosen_index = random.choice(indices)
    sentence = sem_eng_sentences[chosen_index]
    paraphrase_sentence = paraphrases[chosen_index] if chosen_index < len(paraphrases) else "[Paraphrase not found]"

    translations = {"english": sentence, "english_paraphrase": paraphrase_sentence}
    for lang, path in sem_lang_files.items():
        with open(path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f]
            translations[lang] = lines[chosen_index] if chosen_index < len(lines) else "[Index out of range]"

    with open(syn_eng_file, "r", encoding="utf-8") as f:
        syn_sentences = [line.strip() for line in f]
    try:
        syn_id = syn_sentences.index(sentence)
    except ValueError:
        syn_id = None

    with open(group_file, "r", encoding="utf-8") as f:
        group_ids = [line.strip() for line in f]

    group_id = int(group_ids[syn_id]) if syn_id is not None and syn_id < len(group_ids) else "[Group ID not found]"

    return chosen_index, translations, syn_id, group_id


def get_syntax_sentences(group_id, syn_eng_file, group_file, max_lines=5):
    with open(group_file, "r", encoding="utf-8") as f:
        group_ids = [line.strip() for line in f]
    matching_indices = [i for i, gid in enumerate(group_ids) if gid == str(group_id)]
    matching_indices = matching_indices[:max_lines]
    with open(syn_eng_file, "r", encoding="utf-8") as f:
        syn_sentences = [line.strip() for line in f]
    return [syn_sentences[i] for i in matching_indices if i < len(syn_sentences)]

def generate_latex_table(sentence_dict, pos_structure, group_sentences):
    """
    Generate an Overleaf-compatible LaTeX table with 3 columns:
    - Column 1: label (English / Paraphrase / Translations / Syntax etc.)
    - Column 2: sentence(s)
    - Column 3: symbolic representation ($X_i$, $X'_i$, $SEM(X_i)$, $SYN(X_i)$)
    """

    lines = [
        r"\begin{tabular}{|l|p{10cm}|c|}",
        r"\hline"
    ]

    # English sentence
    lines.append(
        f"English & {escape_latex(sentence_dict['english'])} & $X_{{i}}$ \\\\"
    )
    lines.append(r"\hline")

    # POS structure (directly below English)
    lines.append(
        f"POS & {escape_latex(pos_structure)} & \\\\"
    )
    lines.append(r"\hline")

    # Syntax group sentences (also below English)
    if group_sentences:
        first = True
        for s in group_sentences:
            if first:
                lines.append(
                    f"Syntax group & {escape_latex(s)} & $SYN(X_{{i}})$ \\\\"
                )
                first = False
            else:
                lines.append(
                    f" & {escape_latex(s)} & \\\\"
                )
        lines.append(r"\hline")

    # Paraphrase
    lines.append(
        f"Paraphrase & {escape_latex(sentence_dict['english_paraphrase'])} & $X'_{{i}}$ \\\\"
    )
    lines.append(r"\hline")

    # Translations
    first = True
    for lang, text in sentence_dict.items():
        if lang not in ['english', 'english_paraphrase']:
            wrapped_text = wrap_multilingual(text, lang)
            if first:
                lines.append(
                    f"Translations & {lang.capitalize()}: {wrapped_text} & $SEM(X_{{i}})$ \\\\"
                )
                first = False
            else:
                lines.append(
                    f" & {lang.capitalize()}: {wrapped_text} & \\\\"
                )
    lines.append(r"\hline")

    lines.append(r"\end{tabular}")

    return "\n".join(lines)



# --- MAIN ---
if __name__ == "__main__":
    idx, sentences, syn_id, group_id = pick_random_sentence(
        SEM_ENGLISH_FILE, SEM_PARAPHRASE_FILE, SEM_INDICES_FILE, SEM_LANGUAGE_FILES,
        SYN_ENGLISH_FILE, GROUP_IDS_FILE
    )

    # Get POS structure
    pos_structure = ""
    if syn_id is not None:
        with open(SYNTAX_MATCHED_FILE, "r", encoding="utf-8") as f:
            for line in f:
                cols = line.strip().split("\t")
                if len(cols) >= 3 and cols[0] == sentences['english']:
                    pos_structure = cols[2]
                    break

    group_sentences = []
    if group_id != "[Group ID not found]":
        group_sentences = get_syntax_sentences(group_id, SYN_ENGLISH_FILE, GROUP_IDS_FILE, max_lines=5)

    # Generate LaTeX table
    latex_table = generate_latex_table(sentences, pos_structure, group_sentences)

    print(latex_table)


\begin{tabular}{|l|p{10cm}|c|}
\hline
English & We must carefully observe the rules & $X_{i}$ \\
\hline
POS & PRP MD RB VB DT NNS & \\
\hline
Syntax group & You could neatly arrange the items & $SYN(X_{i})$ \\
 & We should fairly distribute the tasks & \\
 & She will kindly assist the newcomers & \\
 & We must carefully observe the rules & \\
 & It can broadly categorize the items & \\
\hline
Paraphrase & The rules must be followed with care & $X'_{i}$ \\
\hline
Translations & Chinese: \begin{CJK}{UTF8}{gbsn}我们必须认真遵守规则\end{CJK} & $SEM(X_{i})$ \\
 & Spanish: Debemos observar cuidadosamente las reglas & \\
 & Italian: Dobbiamo osservare attentamente le regole & \\
 & Turkish: Kurallara dikkatle uymalıyız & \\
 & German: Wir müssen die Regeln sorgfältig beachten & \\
 & Arabic: \RL{ElynA AlAltzAmَ bAlqwAEd bEnAyh} & \\
\hline
\end{tabular}
